<a href="https://colab.research.google.com/github/MorreyB/PC-Free/blob/main/Linux_Virtual_PC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, subprocess, shutil, psutil, time, threading, datetime, socket
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    shutil.rmtree(os.path.join(backup_root, backups[i]))
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & GPU ---
print("፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
    f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)

if not shutil.which("vglrun"):
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run("dpkg -i vgl.deb && apt-get install -f -y", shell=True, capture_output=True)

subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("ldconfig", shell=True)

# --- 4. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if os.path.exists(backup_root):
        backups = sorted([d for d in os.listdir(backup_root)])
        if backups:
            target = os.path.join(backup_root, backups[-1])
            for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
                src = os.path.join(target, name)
                if os.path.exists(src):
                    if os.path.exists(dest): shutil.rmtree(dest)
                    shutil.copytree(src, dest)

# --- 5. CLEAN & START VNC SERVER ---
print("፤ Starting VNC & Proxy...")
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
restore_session()

os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# --- 6. APP LAUNCHERS ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia", "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)

def launch_sober():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.sober.Sober"], env=env)

# --- 7. REFRESH MENU & DISPLAY LINK ---
subprocess.run("update-desktop-database /usr/share/applications", shell=True)
os.makedirs("/root/Desktop", exist_ok=True)
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop

proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
if not proxy_url.endswith('/'): proxy_url += '/'
final_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">\n    <h3 style="margin-top:0; color: #34a853;">Environment Ready</h3>\n    <p>Click below to open your session:</p>\n    <a href="{final_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_link}</a>\n</div>'''))

launch_firefox()
print("፤ All components initialized.")

Mounted at /content/drive
፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL)...


In [ ]:
!bash

bash: cannot set terminal process group (1025): Inappropriate ioctl for device
bash: no job control in this shell


/content# ^C


In [14]:
import socket
import psutil

def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

print(f"፤ Checking VNC Server (5910): {'RUNNING' if check_port(5910) else 'NOT FOUND'}")
print(f"፤ Checking noVNC Proxy (6080): {'RUNNING' if check_port(6080) else 'NOT FOUND'}")

print("\n፤ Active Processes:")
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    if any(x in (proc.info['name'] or '') for x in ['Xtightvnc', 'websockify']):
        print(f" - PID {proc.info['pid']}: {proc.info['name']} ({' '.join(proc.info['cmdline'] or [])})")

፤ Checking VNC Server (5910): NOT FOUND
፤ Checking noVNC Proxy (6080): RUNNING

፤ Active Processes:
 - PID 20946: websockify (/usr/bin/python3 /usr/bin/websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910)


In [15]:
import subprocess, os, time

print("፤ Cleaning up stale VNC locks...")
# Remove existing lock files that might prevent VNC from starting
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10
subprocess.run("pkill -9 -f Xtightvnc", shell=True)

print("፤ Starting VNC Server on :10...")
os.environ['USER'] = 'root'
# Start the VNC server
res = subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True, text=True)
print(res.stdout)

# Verify if it's actually listening now
import socket
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0
    print(f"\n፤ VNC Server Status: {'SUCCESSFULLY STARTED' if is_up else 'FAILED TO START'}")

if not is_up:
    print("፤ Error Log:", res.stderr)

፤ Cleaning up stale VNC locks...
፤ Starting VNC Server on :10...


፤ VNC Server Status: SUCCESSFULLY STARTED


In [3]:
import os, subprocess, shutil, psutil, time, threading, torch, datetime, socket
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    target = os.path.join(backup_root, backups[i])
                    print(f"፤ [Auto-Cleanup] Removing old backup: {backups[i]}")
                    shutil.rmtree(target)
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & GPU DRIVERS ---
print("፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL, Flatpak)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get purge -y firefox", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)
subprocess.run("flatpak remote-add --if-not-exists flathub https://flathub.org/repo/flathub.flatpakrepo", shell=True)

if not shutil.which("vglrun"):
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run("dpkg -i vgl.deb && apt-get install -f -y", shell=True, capture_output=True)

with open('/etc/ld.so.conf.d/00-nvidia-priority.conf', 'w') as f:
    f.write('/usr/lib/x86_64-linux-gnu/nvidia\n/usr/lib64-nvidia\n/usr/lib/x86_64-linux-gnu\n')
subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("ldconfig", shell=True)
for i in range(10):
    if not os.path.exists(f'/dev/nvidia{i}'): subprocess.run(f'mknod -m 666 /dev/nvidia{i} c 195 {i}', shell=True)

# --- 4. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if not os.path.exists(backup_root): return
    backups = sorted([d for d in os.listdir(backup_root) if d != 'latest'])
    if backups:
        target = os.path.join(backup_root, backups[-1])
        print(f"፤ Restoring from: {backups[-1]}")
        for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
            src = os.path.join(target, name)
            if os.path.exists(src):
                if os.path.exists(dest): shutil.rmtree(dest)
                shutil.copytree(src, dest)

# --- 5. START VNC ENGINE ---
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
restore_session()
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# --- 6. APP LAUNCHERS ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia", "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)
    print("፤ Firefox starting...")

def launch_sober():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.sober.Sober"], env=env)
    print("፤ Sober starting...")

final_link = "https://6080-gpu-t4-s-kkb-usw4a0-19ewuilmdlgl5-a.us-west4-0.prod.colab.dev/vnc.html?autoconnect=true&reconnect=true"
display(HTML(f'<div style="padding:15px; border:2px solid #4285f4; border-radius:10px;"><b>Desktop Link:</b> <a href="{final_link}" target="_blank">{final_link}</a></div>'))
print("፤ All components initialized.")

Mounted at /content/drive
፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL, Flatpak)...
፤ Restoring from: 20260619_114346


፤ All components initialized.


In [8]:
import shutil
import subprocess

def ensure_firefox_installed():
    if shutil.which('firefox'):
        print("፤ Firefox is already installed.")
    else:
        print("፤ Firefox not found. Starting installation...")
        # Add PPA for the latest Firefox ESR/Stable
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True)
        subprocess.run("apt-get update", shell=True)
        # Ensure we prioritize the PPA over the snap-stub
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True)
        print("፤ Firefox installation complete.")

ensure_firefox_installed()

፤ Firefox is already installed.


In [11]:
def startup_firefox():
    """
    Ensures Firefox is installed and launches it with VirtualGL on the current desktop session.
    """
    import shutil
    import subprocess
    import os

    # 1. Ensure Installation
    if not shutil.which('firefox'):
        print("፤ Firefox not found. Installing via PPA...")
        subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
        subprocess.run("apt-get update", shell=True, capture_output=True)
        with open('/etc/apt/preferences.d/99mozillateam', 'w') as f:
            f.write('Package: firefox*\nPin: release o=LP-PPA-mozillateam\nPin-Priority: 1001\n')
        subprocess.run("apt-get install -y firefox", shell=True, capture_output=True)
        print("፤ Firefox installation complete.")
    else:
        print("፤ Firefox is already installed.")

    # 2. Launch with GPU Acceleration
    print("፤ Launching Firefox...")
    env = os.environ.copy()
    env.update({
        "DISPLAY": ":10",
        "VGL_DISPLAY": "egl",
        "__GLX_VENDOR_LIBRARY_NAME": "nvidia",
        "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"
    })

    # Use vglrun if available for hardware acceleration
    if os.path.exists("/opt/VirtualGL/bin/vglrun"):
        subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)
    else:
        subprocess.Popen(["firefox", "--no-remote"], env=env)

# Run the startup function
startup_firefox()

፤ Firefox is already installed.
፤ Launching Firefox...


In [16]:
import subprocess
import os

print("፤ Refreshing application menu database...")
# Update the desktop database for the current user
subprocess.run("update-desktop-database /usr/share/applications", shell=True)

# Check if the firefox.desktop file exists
shortcut_path = "/usr/share/applications/firefox.desktop"
if os.path.exists(shortcut_path):
    print(f"፤ Found Firefox shortcut at: {shortcut_path}")
    # Force a refresh of the XFCE panel
    subprocess.run("xfce4-panel --restart", shell=True, env=os.environ.copy())
    print("፤ XFCE Panel restarted. Please check the Application Launcher again.")
else:
    print("፤ Warning: Firefox desktop file not found in /usr/share/applications.")

# Fallback: Create a symlink to the desktop if it's missing from the menu
!ln -sf /usr/share/applications/firefox.desktop /root/Desktop/Firefox.desktop
print("፤ Added a fallback Firefox icon directly to your Desktop.")

፤ Refreshing application menu database...
፤ Found Firefox shortcut at: /usr/share/applications/firefox.desktop
፤ XFCE Panel restarted. Please check the Application Launcher again.
፤ Added a fallback Firefox icon directly to your Desktop.


In [18]:
import subprocess, os, time, socket, google.colab.output
from IPython.display import HTML, display

print("፤ Cleaning stale VNC processes and locks...")
# Kill existing VNC and Websockify
subprocess.run("pkill -9 -f Xtightvnc|websockify", shell=True)
# Remove lock files
!rm -rf /tmp/.X10-lock /tmp/.X11-unix/X10

print("፤ Restarting VNC Server...")
os.environ['USER'] = 'root'
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True, capture_output=True)

print("፤ Restarting noVNC Proxy...")
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)
time.sleep(2)

# Verify connection
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    is_up = s.connect_ex(('127.0.0.1', 5910)) == 0

if is_up:
    proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
    if not proxy_url.endswith('/'): proxy_url += '/'
    new_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

    display(HTML(f'''<div style="padding:20px; border:3px solid #34a853; border-radius:10px; background-color: #f6fdf9;">
        <h3 style="margin-top:0; color: #34a853;">Success: Server Restored</h3>
        <p>Click the link below to reconnect:</p>
        <a href="{new_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{new_link}</a>
    </div>'''))
else:
    print("፤ Critical: VNC Server failed to start. Please check if the 'Master Cell' ran correctly.")

፤ Cleaning stale VNC processes and locks...
፤ Restarting VNC Server...
፤ Restarting noVNC Proxy...


In [ ]:
launch_sober()

፤ Sober starting...


In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# This cell is now redundant and can be removed.

In [ ]:
# To start apps, use the functions defined in the master cell above:
# launch_firefox()
# launch_sober()
# save_session()

In [ ]:
save_session()

፤ Session saved: 20260619_114346


In [2]:
import os
backup_path = '/content/drive/MyDrive/colab_desktop_data/session_backups/'
if os.path.exists(backup_path):
    display(os.listdir(backup_path))
else:
    print('Backup directory not found.')

['20260619_114346']

In [10]:
import google.colab.output

# --- RESTART PROXY ---
# Kill existing proxy if any
!pkill -9 -f websockify

# Start websockify in the background
import subprocess
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# Wait a moment for it to initialize
import time
time.sleep(2)

# Generate the fresh Proxy URL
proxy_url = google.colab.output.eval_js("google.colab.kernel.proxyPort(6080)")
# Ensure there is a trailing slash before vnc.html
if not proxy_url.endswith('/'):
    proxy_url += '/'

final_vnc_link = f"{proxy_url}vnc.html?autoconnect=true&reconnect=true"

from IPython.display import HTML, display
display(HTML(f'''<div style="padding:20px; border:3px solid #4285f4; border-radius:10px; background-color: #f8f9fa;">
    <h3 style="margin-top:0; color: #4285f4;">New Desktop Connection</h3>
    <p>The link has been fixed. Click below to open the desktop:</p>
    <a href="{final_vnc_link}" target="_blank" style="font-weight:bold; font-size:1.1em;">{final_vnc_link}</a>
</div>'''))

In [9]:
launch_firefox()

፤ Firefox starting...


In [6]:
launch_firefox()

፤ Firefox starting...


In [ ]:
import shutil
import os

backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    # Get all subdirectories in the backup folder, excluding any non-timestamped files
    backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])

    if len(backups) > 1:
        oldest_backup = backups[0]
        target_path = os.path.join(backup_root, oldest_backup)
        print(f"፤ Deleting oldest backup: {oldest_backup}")
        shutil.rmtree(target_path)
        print("፤ Deletion complete.")
    else:
        print("፤ Only one or zero backups found. Skipping deletion to prevent data loss.")
else:
    print("፤ Backup directory not found.")

፤ Deleting oldest backup: 20260619_110854
፤ Deletion complete.
